# Behavioral Performance Analysis Pipeline (Reaction Times)

---

**Platforms and Packages Used:**
* **Python Version:** 3.11.8 (Anaconda 24.3.0)
* **NumPy Version:** 1.26.4
* **Pandas Version:** 2.3.2
* **SciPy Version:** 1.13.1
* **Pingouin Version:** 0.5.5

**Terminology Mapping (Code vs. Manuscript):**

| Code / Filename | Manuscript Terminology |
| :--- | :--- |
| `"subject"` | "participant" |
| `"decisions"` | "action conditions" |
| `"stimuli"` | "stimulus types" |

**Overview:**

This notebook processes reaction time (RT) data by aggregating single-trial RTs into subject-level means, and performs within-subject statistical comparisons. Specifically, each Post-association session (Session 1 & Session 2) is compared against the baseline phase (Pre-association) separately for each action condition (Cued vs. Voluntary) and stimulus type (Visual vs. Auditory).

**Pipeline Steps:**
1. **Environmental Setup & Parameters:** Load required packages, define working directories, and set up experimental parameters.
2. **Data Loading & Aggregation:** Compute subject-level mean RTs from single-trial records for all subjects, and specify the distinct participant inclusion lists for univariate and multivariate analyses.
3. **Sample Selection:** Filter participants based on valid trial count thresholds. Generate distinct participant inclusion dictionaries for Univariate (e.g., >= 10 trials) and Multivariate analyses (e.g., >= 40 trials in baseline).
4. **Statistical Assumption Checks:** Evaluate normality using the Shapiro-Wilk test, and verify homogeneity of variance using Levene’s and Bartlett’s tests.
5. **Hypothesis Testing & Effect Sizes:** Executed independently for univariate and multivariate sample sets. Perform paired t-tests (if assumptions are met) or Wilcoxon signed-rank tests (if assumptions are violated), followed by Hedges' g calculation.
6. **Multiple Comparison Correction:** Apply False Discovery Rate (FDR) correction independently for the *Cued* and *Voluntary* conditions.


## Environmental Setup & Parameters

In [1]:
import os
import sys
import scipy
import numpy as np
import pandas as pd
import pingouin as pg
from scipy import stats
from pathlib import Path
from itertools import product

# Set printing and display options
np.set_printoptions(precision=3, linewidth=120)
pd.set_option('display.width', 400)
pd.set_option('display.max_columns', 20)

# Define project directories and paths
# root_dir = Path('/path/to/your/project/directory') # Update with your actual path
root_dir = Path('/home/zhengting/Insync/OneDrive/ParisCite/1_MultivariateProcedure/code/Github_pub/')
stat_dir = root_dir / 'statdata'
mat_dir  = root_dir / 'matdata'

os.makedirs(stat_dir, exist_ok=True)

# Define experimental factors and conditions
num_subs = 20
# days = ['Day1', 'Day2']
sessions = ['Session1', 'Session2']
decisions = ['Cued', 'Voluntary']
stimuli = ['Visual', 'Auditory']
analyses = ['Univariate','Multivariate']

## Data Loading & Aggregation

In [2]:
## get infos of RT from matdata saved in AEP_01_EEG_preprocessing
# RT_infos =  scipy.io.loadmat(mat_dir/'RT_infos.mat')['RT_subs']
RT_infos =  scipy.io.loadmat(mat_dir/'RT_infos_update.mat')['RT_subs_update']

# Initialize an empty list to compile summary statistics
agg_behavior_data = []

# Loop through conditions to load and aggregate single-trial behavioral metrics

for sid in np.arange(1, num_subs+1):
    # info = RT_infos[0,sid-1]['RT_infos']
    info = RT_infos[0,sid-1]['RT']

    for idx, trialtype in enumerate(info['TrialType'].squeeze()):
        # count = info[idx]['RT_final'][0].shape[1]
        # ratio = count/info[idx]['RT_original'][0].shape[1]
        # mean_rt = info[idx]['RT_Noout_Across'][0].mean()
        # std_rt = info[idx]['RT_Noout_Across'][0].std()
        count = info[idx]['RT_Noout_Across'][0].shape[1]
        ratio = count/info[idx]['RT_original'][0].shape[1]
        mean_rt = info[idx]['RT_Noout_Across'][0].mean()+0.150
        std_rt = (info[idx]['RT_Noout_Across'][0]+0.150).std()
        
        agg_behavior_data.append({
            'Subject': sid,
            'TrialType': trialtype[0],
            'MeanRT': mean_rt*1000, # unit: ms
            'STDRT': std_rt*1000, # unit: ms
            'Count': count,
            'Ratio': ratio
        })

# Convert the compiled list into a structured master dataframe
df_behavior_master = pd.DataFrame(agg_behavior_data)
df_behavior_meanrt = df_behavior_master.pivot(index='Subject', columns='TrialType', values='MeanRT')
df_behavior_stdrt = df_behavior_master.pivot(index='Subject', columns='TrialType', values='STDRT')
df_behavior_count  = df_behavior_master.pivot(index='Subject', columns='TrialType', values='Count')
df_behavior_ratio  = df_behavior_master.pivot(index='Subject', columns='TrialType', values='Ratio')

with pd.ExcelWriter(stat_dir/'Behavioral_subjects.xlsx', engine='openpyxl') as writer:
    df_behavior_meanrt.to_excel(writer, sheet_name='MeanRT', index=False)
    df_behavior_stdrt.to_excel(writer, sheet_name='STDRT', index=False)
    df_behavior_count.to_excel(writer, sheet_name='Count', index=False)
    df_behavior_ratio.to_excel(writer, sheet_name='Ratio', index=False)


## Sample Selection

In [3]:
# Participant samples inclusion dictionary (matching the CSD analysis criteria)

thres_count_univariate = 10
thres_count_multivariate = 40
dict_samples = {'Univariate': {}, 'Multivariate': {}}
for decis, stim in product(decisions, stimuli):

    relevant_cols = [col for col in df_behavior_count.columns if decis in col and stim in col]
    pre_cols = [col for col in relevant_cols if 'Pre' in col]

    if relevant_cols:
        mask_uni = (df_behavior_count[relevant_cols] >= thres_count_univariate).all(axis=1)
        dict_samples['Univariate'][f'{decis}_{stim}'] = df_behavior_count.index[mask_uni].to_numpy()
    else:
        dict_samples['Univariate'][f'{decis}_{stim}'] = np.array([])

    if pre_cols:
        mask_multi = (df_behavior_count[pre_cols] >= thres_count_multivariate).all(axis=1)
        ## Only compare PreAssn count with the threshold in multivariate analysis, because it only uses 
        ## the pre-action period. PostAssn data contains both WithStim and NoStim trials and definitely exceeds the threshold.
        dict_samples['Multivariate'][f'{decis}_{stim}'] = df_behavior_count.index[mask_multi].to_numpy()
    else:
        dict_samples['Multivariate'][f'{decis}_{stim}'] = np.array([])

dict_samples

{'Univariate': {'Cued_Visual': array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]),
  'Cued_Auditory': array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]),
  'Voluntary_Visual': array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 16, 17, 18, 19, 20]),
  'Voluntary_Auditory': array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 16, 17, 18, 19, 20])},
 'Multivariate': {'Cued_Visual': array([ 1,  3,  4,  5,  6,  7,  8,  9, 10, 12, 14, 15, 16, 17, 18, 20]),
  'Cued_Auditory': array([ 1,  2,  3,  5,  6,  7,  8,  9, 10, 11, 12, 14, 16, 17, 18, 19, 20]),
  'Voluntary_Visual': array([ 1,  5,  6,  9, 11, 13, 14, 16, 17, 20]),
  'Voluntary_Auditory': array([ 1,  4,  6,  9, 10, 13, 14, 16, 17])}}

## Statistical Assumption, Tests & Correction

In [17]:
# Loop through the two independent analysis pipelines (Univariate and Multivariate)
for analysis_type in analyses:
    print(f"\n" + "="*85)
    print(f" PIPELINE: {analysis_type.upper()} SAMPLE SET")
    print("="*85)
    
    current_sids_dict = dict_samples[analysis_type]
    
    # Perform structured comparisons for each condition
    for decis in decisions:
        print(f"\n--- Decision Condition: {decis.upper()} (Independent FDR Correction) ---")
        stat_results_pool = []
        for stim in stimuli:
            included_sids = current_sids_dict[f'{decis}_{stim}']
            
            # Isolate data subset for the current condition AND current sample set
            df_count_included = df_behavior_count.loc[included_sids]
            df_meanrt_included = df_behavior_meanrt.loc[included_sids]
    
            # Compare each post-association session against the baseline phase
            # for day in days:
            for post_sess in sessions:
                # rt_baseline = df_meanrt_included[f'Day1_{decision}No{stimulus}StimPre'].values
                # post_trial_1 = f'{day}_{decis}No{stim}StimPost'
                # post_trial_2 = f'{day}_{decis}{stim}'
                rt_baseline = df_meanrt_included[f'Session1_{decis}No{stim}StimPre'].values
                post_trial_1 = f'{post_sess}_{decis}No{stim}StimPost'
                post_trial_2 = f'{post_sess}_{decis}{stim}'
                rt_post = (df_count_included[post_trial_1].values*df_meanrt_included[post_trial_1].values+
                           df_count_included[post_trial_2].values*df_meanrt_included[post_trial_2].values)/(
                           df_count_included[post_trial_1].values+df_count_included[post_trial_2].values)
                paired_diff = rt_post - rt_baseline
        
                # 1. Assumption Checks: Normality (Shapiro) & Homogeneity (Levene/Bartlett)
                normality_res = pg.normality(paired_diff, method='shapiro')
                is_normal = normality_res['normal'].values[0]
                shapiro_p = normality_res['pval'].values[0]
                
                _, levene_p = stats.levene(rt_post, rt_baseline)
                _, bartlett_p = stats.bartlett(rt_post, rt_baseline)
                is_homogenous = (levene_p > 0.05) and (bartlett_p > 0.05)
                
                # 2. Hypothesis Testing: T-test if assumptions met, otherwise Wilcoxon
                if is_normal and is_homogenous:
                    test_res = pg.ttest(rt_post, rt_baseline, paired=True, alternative='two-sided')
                    p_uncorrected = test_res['p-val'].values[0]
                    test_stat = test_res['T'].values[0]
                    test_type = 'Paired t-test'
                else:
                    test_res = pg.wilcoxon(rt_post, rt_baseline, alternative='two-sided')
                    p_uncorrected = test_res['p-val'].values[0]
                    test_stat = test_res['W-val'].values[0]
                    test_type = 'Wilcoxon test'
                    
                # 3. Effect Size: Hedges' g
                hedges_g = pg.compute_effsize(rt_post, rt_baseline, paired=True, eftype='hedges')
                
                # Append outputs
                stat_results_pool.append({
                    'Condition': f"{decis}_{stim}",
                    'Comparison': f"{post_sess} vs PreAssn",
                    'PostmeanRT': rt_post.mean(),
                    'PremeanRT': rt_baseline.mean(),
                    'PostSTDRT': rt_post.std(),
                    'PreSTDRT': rt_baseline.std(),
                    'Test_Type': test_type,
                    'Stat_Val': test_stat,
                    'Shap_p': shapiro_p,
                    'Lev_p': levene_p,
                    'Bart_p': bartlett_p,
                    'p_unc': p_uncorrected,
                    'Hedges_g': hedges_g
                })
    
        # Convert to dataframe for current analysis type
        df_stats_summary = pd.DataFrame(stat_results_pool)

        # 4. Multiple Comparison Correction: Independent FDR correction for this sample branch
        reject, p_corrected = pg.multicomp(df_stats_summary['p_unc'].values, method='fdr_bh')
        df_stats_summary['p_fdr'] = p_corrected
        df_stats_summary['Sign'] = reject
    
        # Display results
        print(df_stats_summary.to_string(index=False, formatters={
            'PostmeanRT': '{:0.0f}'.format,
            'PremeanRT': '{:0.0f}'.format,
            'PostSTDRT': '{:0.0f}'.format,
            'PreSTDRT': '{:0.0f}'.format,
            'Stat_Val': '{:0.2f}'.format,
            'Shap_p': '{:0.2f}'.format,
            'Lev_p': '{:0.2f}'.format,
            'Bart_p': '{:0.2f}'.format,
            'p_unc': '{:0.4f}'.format,
            'p_fdr': '{:0.4f}'.format,
            'Hedges_g': '{:0.3f}'.format
        }))
        print("\n")


 PIPELINE: UNIVARIATE SAMPLE SET

--- Decision Condition: CUED (Independent FDR Correction) ---
    Condition          Comparison PostmeanRT PremeanRT PostSTDRT PreSTDRT     Test_Type Stat_Val Shap_p Lev_p Bart_p  p_unc Hedges_g  p_fdr  Sign
  Cued_Visual Session1 vs PreAssn        564       552        59       62 Paired t-test     1.04   0.86  0.57   0.81 0.3103    0.190 0.4138 False
  Cued_Visual Session2 vs PreAssn        528       552        78       62 Paired t-test    -1.54   0.78  0.51   0.34 0.1409   -0.325 0.2818 False
Cued_Auditory Session1 vs PreAssn        587       566        63       62 Paired t-test     1.90   0.71  0.98   0.96 0.0731    0.326 0.2818 False
Cued_Auditory Session2 vs PreAssn        558       566        88       62 Paired t-test    -0.48   0.22  0.20   0.14 0.6358   -0.098 0.6358 False



--- Decision Condition: VOLUNTARY (Independent FDR Correction) ---
         Condition          Comparison PostmeanRT PremeanRT PostSTDRT PreSTDRT     Test_Type Stat_Val S